# R02 · Partition structure (Tables 7, 8, S2; Sections 4.2, 4.5, 4.6)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/<your-account>/nids-partition-distortion-ae-dae/blob/main/notebooks/reproduction/R02_Partition_Structure.ipynb)

Recomputes total-variation distances, gate accounting, solver status, the provenance hash chain and the CICIDS2017 integrity summary.

**How to run:** Runtime → Run all. The first run in a fresh Colab runtime installs the pinned library versions and restarts the runtime once; then choose Run all again.

<details><summary>Source and conversion notes</summary>

Reproduce the partition-structure results (manuscript Sections 4.2, 4.5, 4.6; Tables 7, 8, S2).

Inputs (all inside the repository): the UNSW-NB15 v3.4 gate tables, integrity report, amendment and
solver manifests; the source-balanced fingerprint-level development summary; the smoke manifests; the
CICIDS2017 integrity report and allocation tables; the first 100,000 rows of each membership file;
the development/final run summaries derived from the prediction archives.
Outputs:
  results/tables/Table_3_2_UNSW_structural_validation.csv   (manuscript Table 8)
  results/tables/Table_3_2b_source_balanced_profile.csv     (manuscript Table 7)
  results/tables/Table_3_3_ge100_band_extract.csv           (manuscript Table S2, >=100 band)
  results/tables/Table_S2_role_by_band_allocation.csv       (manuscript Table S2, all bands)
  results/tables/Table_3_2c_role_tv_distances.csv           (TV distance of every v3.4 role)
  results/claims_check/reproduce_partitions.csv
Rows that come from the analysis record rather than an artefact are kept in the tables and labelled so.

Mechanical changes made when copying the .py files into this notebook:
`from src... import ...` lines are removed (the library code is defined in the cells above);
`sys.path.insert(...)` and `from __future__ import annotations` are removed;
`Path(__file__)...` repository roots are replaced by `REPO_ROOT` from the setup cell;
the command-line interface of `src/utils/hashing.py` is omitted;
command-line arguments of `reproduce_final_statistics.py` are replaced by the parameter cell;
`sys.exit(L.finish())` becomes `assert L.finish() == 0`.
</details>

In [ ]:
# ===== Setup: run this cell first =====
# REPO_SOURCE = "drive": the repository is in your Google Drive, either as the unzipped folder
#   `nids-partition-distortion-ae-dae` or as the zip file `nids-partition-distortion-ae-dae.zip`, anywhere
#   in My Drive (up to four folders deep). The cell finds it; set DRIVE_REPO only to point at one copy.
# REPO_SOURCE = "github": after the repository is on GitHub (fill in GITHUB_URL).
REPO_SOURCE = "drive"                                                    # "drive" or "github"
DRIVE_REPO = "/content/drive/MyDrive/nids-partition-distortion-ae-dae"   # optional hint
GITHUB_URL = "https://github.com/<your-account>/nids-partition-distortion-ae-dae.git"

import os, subprocess, sys, zipfile
from importlib.metadata import version
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_NAME = "nids-partition-distortion-ae-dae"


def _is_repo(p: Path) -> bool:
    return (p / "artifacts" / "ARTIFACT_PROVENANCE.csv").is_file()


def _find_repo(start: Path) -> Path:
    for p in [start, *start.parents]:
        if _is_repo(p):
            return p
    raise FileNotFoundError(f"repository folder not found above {start}")


def _locate_in_drive(hint: Path, drive_root: Path) -> Path:
    # 1. the hint itself, a folder above it, or a nested copy below it (unzipping often creates name/name/)
    for c in [hint, *hint.parents, hint / REPO_NAME]:
        if _is_repo(c):
            return c
    # 2. the unzipped folder anywhere in My Drive, up to four levels deep
    for depth in range(0, 4):
        pattern = "/".join(["*"] * depth + ["artifacts", "ARTIFACT_PROVENANCE.csv"])
        hits = sorted(drive_root.glob(pattern))
        if hits:
            if len(hits) > 1:
                print("Several copies found; using the first (set DRIVE_REPO to choose):", *[h.parents[1] for h in hits], sep="\n  ")
            return hits[0].parents[1]
    # 3. the zip file anywhere in My Drive (Google Drive does not unzip uploads): extract it to /content
    for depth in range(0, 4):
        zips = sorted(drive_root.glob("/".join(["*"] * depth + [REPO_NAME + "*.zip"])))
        if zips:
            dest = Path("/content/repo_from_zip")
            print(f"Found {zips[0]} - extracting to {dest} (results will be written there, not to Drive)")
            with zipfile.ZipFile(zips[0]) as z:
                z.extractall(dest)
            return _locate_in_drive(dest, dest)
    near = [p.name for p in drive_root.iterdir() if "nids" in p.name.lower() or "partition" in p.name.lower()] if drive_root.exists() else []
    raise FileNotFoundError(
        "Could not find the repository in Google Drive.\n"
        f"Looked for '{REPO_NAME}/artifacts/ARTIFACT_PROVENANCE.csv' (up to 4 folders deep) and for '{REPO_NAME}*.zip' in {drive_root}.\n"
        f"Items in My Drive whose names look related: {near or 'none'}.\n"
        "Fix: upload the zip file (or the unzipped folder) to My Drive, or set DRIVE_REPO to the folder that "
        "contains README.md, artifacts/ and notebooks/.")


_previous = globals().get("REPO_ROOT")                 # set by an earlier notebook in the same runtime (R00)
if _previous is not None and _is_repo(Path(_previous)):
    REPO_ROOT = Path(_previous)
elif IN_COLAB and REPO_SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_ROOT = _locate_in_drive(Path(DRIVE_REPO), Path("/content/drive/MyDrive"))
elif IN_COLAB:
    target = Path("/content/" + REPO_NAME)
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1", GITHUB_URL, str(target)], check=True)
    REPO_ROOT = _find_repo(target)
else:                                     # local Jupyter: the notebook lives in notebooks/reproduction/
    REPO_ROOT = _find_repo(Path.cwd().resolve())
os.chdir(REPO_ROOT)

# The results were verified with these exact versions (the versions recorded by the Colab analysis).
PINNED = {"numpy": "2.1.3", "pandas": "2.2.3", "scipy": "1.16.3", "scikit-learn": "1.6.1",
          "matplotlib": "3.10.0", "pytest": "8.3.5"}
wrong = {k: v for k, v in PINNED.items() if version(k) != v}
if wrong and IN_COLAB:
    print("Installing the pinned versions:", wrong)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_ROOT / "requirements.txt")], check=True)
    print("Done. The runtime will now restart; then run the notebook again from the top (Runtime > Run all).")
    os.kill(os.getpid(), 9)
elif wrong:
    print("WARNING: library versions differ from requirements.txt:", wrong)
print("Repository:", REPO_ROOT)
print("Python", sys.version.split()[0], "|", ", ".join(f"{k} {version(k)}" for k in PINNED))

## Library code
Verbatim copies of the repository modules this notebook uses (`src/`). Run these cells; there is nothing to edit.

In [ ]:
# ===== src/utils/io_utils.py =====
"""Paths and loaders shared by the reproduction scripts and tests."""

import json
import os
from pathlib import Path

import pandas as pd

REPO_ROOT = Path(REPO_ROOT)  # set by the setup cell
ARTIFACTS = REPO_ROOT / "artifacts"
RESULTS = REPO_ROOT / "results"
DATASETS = {"cicids2017": "CICIDS2017", "unsw_nb15": "UNSW-NB15"}

# Columns of every observation-level prediction archive written by NB-03/NB-07.
ARCHIVE_COLUMNS = ["row_id", "y_true", "reconstruction_error", "threshold", "y_pred", "model", "seed",
                   "config_hash", "run_hash", "attack_family", "feature_fingerprint"]
ARCHIVE_DTYPES = {"row_id": str, "y_true": "int8", "reconstruction_error": "float64", "threshold": "float64",
                  "y_pred": "int8", "model": str, "seed": "int64", "config_hash": str, "run_hash": str,
                  "attack_family": str, "feature_fingerprint": str}
# CICIDS2017 web-attack family names contain U+FFFD where the original CSVs had a cp1252 en dash
# ("Web Attack � Brute Force"); the character is kept as stored in the archives.

# Environment variable that points to the folder holding the external prediction archives.
ARCHIVE_ENV = "NIDS_PREDICTION_ARCHIVES"


def art(*parts: str) -> Path:
    """Path inside ``artifacts/``."""
    return ARTIFACTS.joinpath(*parts)


def load_json(path: str | Path) -> dict:
    with open(path, encoding="utf-8") as fh:
        return json.load(fh)


def read_csv(path: str | Path, **kw) -> pd.DataFrame:
    """``pandas.read_csv`` with exact (round-trip) float parsing, so that committed tables compare exactly."""
    kw.setdefault("float_precision", "round_trip")
    return pd.read_csv(path, **kw)


def read_prediction_archive(path: str | Path) -> pd.DataFrame:
    """Read one observation-level prediction archive with the dtypes used in the study."""
    return pd.read_csv(path, usecols=ARCHIVE_COLUMNS, dtype=ARCHIVE_DTYPES)


def archive_dir(cli_value: str | None = None) -> Path | None:
    """Folder with the external prediction archives (CLI argument first, then the environment variable)."""
    value = cli_value or os.environ.get(ARCHIVE_ENV)
    if not value:
        return None
    p = Path(value).expanduser()
    return p if p.is_dir() else None


def dataset_of_row_id(row_id: str) -> str:
    """``row_id`` is ``<dataset>::<source file>::<10-digit source row>``; return the repository dataset key."""
    prefix = row_id.split("::", 1)[0]
    return {"CICIDS2017": "cicids2017", "UNSWNB15": "unsw_nb15"}[prefix]


def role_sizes(ds: str) -> dict:
    """Evaluation-role sizes recorded in the frozen result tables (TP+FP+FN+TN of any run)."""
    fin = read_csv(art(ds, "final_evaluation", "final_seed_results.csv")).iloc[0]
    dev = read_csv(art(ds, "model_selection", "confirmation_seed_results.csv")).iloc[0]
    tot = lambda r: int(r.TP + r.FP + r.FN + r.TN)
    return {"final_test": tot(fin), "development": tot(dev)}


def write_csv(df: pd.DataFrame, path: str | Path, **kw) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, lineterminator="\n", **kw)
    return path

In [ ]:
# ===== src/utils/claims.py =====
"""Record manuscript claims next to the values recomputed from the artefacts.

Each check compares a recomputed number with the value *as printed in the manuscript*. The tolerance is
half a unit in the last printed decimal place, so "0.7094" passes for any value in [0.70935, 0.70945].
Every reproduction script writes its log to ``results/claims_check/<script>.csv`` and exits with a
non-zero status if any check fails.
"""

import math
import sys
from pathlib import Path

import pandas as pd



def _parse(printed: str) -> tuple[float, float]:
    s = str(printed).replace(",", "").replace("%", "").replace("+", "").replace("−", "-").strip()
    dec = len(s.split(".")[1]) if "." in s and "e" not in s.lower() else 0
    if "e" in s.lower():
        mant = s.lower().split("e")[0]
        exp = int(s.lower().split("e")[1])
        dec_m = len(mant.split(".")[1]) if "." in mant else 0
        return float(s), 0.5 * 10 ** (exp - dec_m)
    return float(s), 0.5 * 10 ** (-dec)


class ClaimLog:
    def __init__(self, script: str):
        self.script = script
        self.rows: list[dict] = []

    def check(self, claim_id: str, location: str, description: str, reproduced, printed: str,
              source: str, scale: float = 1.0) -> bool:
        """``scale`` converts the reproduced value to the printed unit (e.g. 100 for percentages)."""
        value, tol = _parse(printed)
        rep = float(reproduced) * scale
        ok = math.isclose(rep, value, abs_tol=tol + 1e-12)
        self.rows.append({"claim_id": claim_id, "manuscript_location": location, "description": description,
                          "manuscript_value": printed, "reproduced_value": repr(round(rep, 10)),
                          "status": "PASS" if ok else "FAIL", "source": source})
        return ok

    def assert_true(self, claim_id: str, location: str, description: str, condition: bool, source: str) -> bool:
        self.rows.append({"claim_id": claim_id, "manuscript_location": location, "description": description,
                          "manuscript_value": "true", "reproduced_value": str(bool(condition)),
                          "status": "PASS" if condition else "FAIL", "source": source})
        return bool(condition)

    def note(self, claim_id: str, location: str, description: str, value: str, source: str, status: str) -> None:
        """Record a finding that is not a pass/fail comparison (e.g. a discrepancy that the manuscript notes)."""
        self.rows.append({"claim_id": claim_id, "manuscript_location": location, "description": description,
                          "manuscript_value": "", "reproduced_value": value, "status": status, "source": source})

    def finish(self) -> int:
        df = pd.DataFrame(self.rows)
        out = RESULTS / "claims_check" / f"{self.script}.csv"
        out.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(out, index=False, lineterminator="\n")
        n_fail = int((df.status == "FAIL").sum())
        n_pass = int((df.status == "PASS").sum())
        print(f"[{self.script}] {n_pass} checks passed, {n_fail} failed, {len(df) - n_pass - n_fail} notes -> {out}")
        if n_fail:
            print(df[df.status == "FAIL"].to_string(index=False), file=sys.stderr)
        return 1 if n_fail else 0

In [ ]:
# ===== src/fingerprinting/multiplicity_audit.py =====
"""Fingerprint-multiplicity bands, total-variation distance and related structural quantities.

The eight frozen bands are those of the UNSW-NB15 source-multiplicity amendment
(``source_multiplicity_amendment.json["bands"]``): 1, 2, 3-4, 5-9, 10-19, 20-49, 50-99, >=100.
"""

import numpy as np
import pandas as pd

BANDS = ["1", "2", "3-4", "5-9", "10-19", "20-49", "50-99", ">=100"]
EDGES = [0, 1, 2, 4, 9, 19, 49, 99, np.inf]   # right-closed intervals (0,1], (1,2], (2,4], ...


def band_of(group_size) -> pd.Series:
    """Map fingerprint-group sizes (multiplicities) to the frozen band labels."""
    return pd.cut(pd.Series(group_size), bins=EDGES, labels=BANDS, right=True).astype(str)


def shares(rows_by_band: pd.Series) -> pd.Series:
    """Band shares of rows, in the frozen band order."""
    r = rows_by_band.reindex(BANDS).fillna(0).astype(float)
    return r / r.sum()


def tv_distance(p: pd.Series, q: pd.Series) -> float:
    """Total-variation distance 1/2 * sum_b |p_b - q_b| between two band-share vectors."""
    return float(0.5 * (shares(p) - shares(q)).abs().sum())


def gate_rows(gate: pd.DataFrame, column: str = "actual_rows") -> pd.DataFrame:
    """Role x band row matrix from ``multiplicity_gate.csv``."""
    return gate.pivot(index="partition", columns="size_band", values=column)[BANDS]


def expected_cross_side_sharing(group_sizes: np.ndarray) -> float:
    """Expected number of rows sharing their fingerprint with >=1 row on the opposite side when every row
    is assigned independently to one of two sides with probability 1/2: sum_g m_g (1 - 2^{-(m_g - 1)})."""
    m = np.asarray(group_sizes, dtype=float)
    return float((m * (1.0 - np.power(0.5, m - 1.0))).sum())


def redundancy_rate(rows: int, groups: int) -> float:
    """(rows - groups) / rows: the share of rows that repeat a model-view vector already present."""
    return (rows - groups) / rows

In [ ]:
# ===== src/partitioning/partition_integrity.py =====
"""Integrity and accounting checks on the frozen partition artefacts.

The allocation itself (fingerprint-atomic heuristic for CICIDS2017; source- and multiplicity-aware
MILP for UNSW-NB15) needs the full membership and raw data and is not re-run here. These functions
re-check the properties the manuscript reports, using the frozen gate tables, the integrity reports
and the first 100,000 rows of each membership file.
"""

import pandas as pd


BENIGN_ONLY_ROLES = ("benign_weight_train", "benign_early_stop", "benign_threshold_calibration")
EVALUATION_ROLES = ("development", "final_test")
ROLES = BENIGN_ONLY_ROLES + EVALUATION_ROLES


def membership_checks(m: pd.DataFrame) -> dict:
    """Checks on a membership table (full file or the first-100k sample)."""
    states = m.groupby("feature_fingerprint")["partition"].nunique()
    benign_only = m[m.partition.isin(BENIGN_ONLY_ROLES)]
    mixed = m[m.fingerprint_group_class.str.startswith("mixed")]
    return {
        "rows": len(m),
        "row_id_unique": bool(m.row_id.is_unique),
        "fingerprints_in_more_than_one_state": int((states > 1).sum()),
        "attack_rows_in_benign_only_roles": int((benign_only.binary_label != 0).sum()),
        "mixed_group_rows_in_benign_only_roles": int(benign_only.fingerprint_group_class.str.startswith("mixed").sum()),
        "mixed_group_states": sorted(mixed.partition.unique().tolist()),
    }


def role_capacity_checks(gate: pd.DataFrame, integrity: dict) -> dict:
    """UNSW-NB15 v3.4: gate role totals versus the capacities in the integrity report."""
    totals = gate.groupby("partition").actual_rows.sum()
    cap = integrity["role_pure_benign_capacities"]
    return {"role_totals_equal_capacities": all(int(totals[r]) == int(cap[r]) for r in cap),
            "total_equals_pure_benign_rows": int(totals.sum()) == integrity["pure_benign_rows"],
            "all_cells_pass": bool(gate["pass"].all()),
            "max_abs_deviation": int(gate.absolute_deviation.max()),
            "cells_at_max_deviation": gate.loc[gate.absolute_deviation == gate.absolute_deviation.max(),
                                               ["partition", "size_band", "absolute_deviation", "tolerance"]]
            .to_dict("records")}


def source_checks(source_actual: pd.DataFrame, singleton_gate: pd.DataFrame, sxm: pd.DataFrame,
                  gate: pd.DataFrame) -> dict:
    """Cross-table consistency of the UNSW-NB15 source gates with the multiplicity gate."""
    by_role = source_actual.set_index("partition").sum(axis=1)
    g_role = gate.groupby("partition").actual_rows.sum()
    sing = singleton_gate.groupby("partition").actual_rows.sum()
    g_sing = gate[gate.size_band == "1"].set_index("partition").actual_rows
    sxm_rb = sxm.groupby(["partition", "size_band"]).actual_rows.sum()
    g_rb = gate.set_index(["partition", "size_band"]).actual_rows
    nonexempt = sxm[~sxm.is_exempt]
    return {"source_totals_equal_gate_totals": bool((by_role.reindex(g_role.index) == g_role).all()),
            "singleton_source_equals_gate_band1": bool((sing.reindex(g_sing.index) == g_sing).all()),
            "source_x_band_sums_equal_gate": bool((sxm_rb.reindex(g_rb.index) == g_rb).all()),
            "nonexempt_cells": len(nonexempt), "nonexempt_cells_pass": bool(nonexempt.pass_nonexempt.all()),
            "exempt_cells": sxm.loc[sxm.is_exempt, ["source_file", "size_band"]].drop_duplicates().to_dict("records")}


def band_ge100_checks(band_report: pd.DataFrame, gate: pd.DataFrame) -> dict:
    g = gate[gate.size_band == ">=100"].set_index("partition").actual_rows
    b = band_report.set_index("partition")
    return {"rows_equal_gate": bool((b.rows.reindex(g.index) == g).all()),
            "groups_total": int(b.groups.sum()), "rows_total": int(b.rows.sum()),
            "largest_group_by_role": b.max_group_size.astype(int).to_dict()}


def cicids_integrity_summary(report: dict) -> dict:
    checks = report["hard_validation_results"]
    return {"status": report["final_status"], "hard_checks": len(checks),
            "hard_checks_passed": sum(bool(c["passed"]) for c in checks if c["hard"]),
            "blocking_failures": len(report["blocking_failures"]),
            "development_attack_prevalence": report["observed_allocation"]["development_attack_prevalence"],
            "final_test_attack_prevalence": report["observed_allocation"]["final_test_attack_prevalence"],
            "attack_reserve_rows": report["observed_allocation"]["attack_reserve_rows"]}

## Reproduction: `scripts/reproduce_partitions.py`

In [ ]:
import sys
from pathlib import Path


import pandas as pd  # noqa: E402


L = ClaimLog("reproduce_partitions")
F = ("unsw_nb15", "partition", "final_source_multiplicity_balanced")
SB = ("unsw_nb15", "partition", "source_balanced")
REC = "analysis record (no artefact in the evidence set)"

gate = read_csv(art(*F, "multiplicity_gate.csv"))
integ = load_json(art(*F, "partition_integrity_report.json"))
band = read_csv(art(*F, "band_ge100_role_report.csv"))
rows = gate_rows(gate)
pool = rows.sum(axis=0)

### source-balanced allocation (Table 7)

In [ ]:
fs = read_csv(art(*SB, "development_benign_fingerprint_level_summary.csv"))
pure = fs[fs.fingerprint_group_class == "pure benign group"].copy()
pure["band"] = band_of(pure.benign_rows).to_numpy()
dev_sb = pure.groupby("band").benign_rows.sum().reindex(BANDS).fillna(0)
tv_sb = tv_distance(dev_sb, pool)
ge100_sb = pure[pure.band == ">=100"]
L.check("T7.dev.singleton_share", "Table 7", "development singleton share (source-balanced)", 100 * dev_sb["1"] / dev_sb.sum(), "13.93", "development_benign_fingerprint_level_summary.csv")
L.check("T7.dev.repeated_share", "Sec. 4.2", "development rows in repeated fingerprints (%)", 100 * (1 - dev_sb["1"] / dev_sb.sum()), "86.07", "as above")
L.check("T7.dev.ge100_rows", "Table 7; Sec. 4.2", "development rows in the >=100 band", dev_sb[">=100"], "39,202", "as above")
L.check("T7.dev.ge100_pct", "Table 7", "share of development rows in the >=100 band", 100 * dev_sb[">=100"] / dev_sb.sum(), "17.75", "as above")
L.check("T7.dev.ge100_groups", "Sec. 4.2", "development groups in the >=100 band", len(ge100_sb), "222", "as above")
L.check("T7.pool.singleton_share", "Table 7; Sec. 4.5", "pool singleton share", 100 * pool["1"] / pool.sum(), "84.87", "multiplicity_gate.csv")
L.check("T7.pool.ge100_rows", "Table 7", "pool rows in the >=100 band", pool[">=100"], "51,479", "multiplicity_gate.csv")
L.check("T7.pool.ge100_pct", "Table 7", "pool share in the >=100 band", 100 * pool[">=100"] / pool.sum(), "2.32", "multiplicity_gate.csv")
L.check("T8.sourcebal.tv", "Table 8; abstract; Sec. 4.2", "TV(development, pool), source-balanced", tv_sb, "0.7094", "as above + multiplicity_gate.csv")
tgt = gate.pivot(index="partition", columns="size_band", values="target_rows")[BANDS]
L.check("S4.2.dev_target_ge100", "Sec. 4.2", "proportional development target in the >=100 band", tgt.loc["development", ">=100"], "5,127", "multiplicity_gate.csv")
L.check("S4.2.wt_target_ge100", "Sec. 4.2", "proportional weight-training target in the >=100 band", tgt.loc["benign_weight_train", ">=100"], "25,751", "multiplicity_gate.csv")
L.note("T7.other_roles", "Table 7", "singleton shares 100.00/47.49/96.09/95.23% and >=100 shares of the other four roles in the source-balanced allocation",
       "not recomputable: the source-balanced membership file is not in the evidence set", REC, "NOT_REPRODUCIBLE_HERE")
L.note("S4.2.feasibility", "Sec. 4.2", "multiplicity-only feasibility bound 3.94e-6 (one-row aggregate mismatch)",
       "not recomputable here", REC, "NOT_REPRODUCIBLE_HERE")
t7 = pd.DataFrame([
    {"role": "benign_weight_train", "singleton_share_pct": "100.00", "ge100_rows": "0 (0.00%)", "source": REC},
    {"role": "benign_early_stop", "singleton_share_pct": "47.49", "ge100_rows": "5.53% of role", "source": REC},
    {"role": "benign_threshold_calibration", "singleton_share_pct": "96.09", "ge100_rows": "0 (0.00%)", "source": REC},
    {"role": "development", "singleton_share_pct": f"{100 * dev_sb['1'] / dev_sb.sum():.2f}",
     "ge100_rows": f"{int(dev_sb['>=100']):,} ({100 * dev_sb['>=100'] / dev_sb.sum():.2f}%)",
     "source": "recomputed: source_balanced/development_benign_fingerprint_level_summary.csv"},
    {"role": "final_test", "singleton_share_pct": "95.23", "ge100_rows": "0 (0.00%)", "source": REC},
    {"role": "pure-benign pool", "singleton_share_pct": f"{100 * pool['1'] / pool.sum():.2f}",
     "ge100_rows": f"{int(pool['>=100']):,} ({100 * pool['>=100'] / pool.sum():.2f}%)",
     "source": "recomputed: final_source_multiplicity_balanced/multiplicity_gate.csv"}])
write_csv(t7, RESULTS / "tables" / "Table_3_2b_source_balanced_profile.csv")

### v3.4 allocation (Sections 4.5, S2)

In [ ]:
gates = integ["gates"]
L.assert_true("S4.5.all_gates", "Sec. 4.5", "integrity report: PASS, every gate true and zero fingerprint crossings",
              integ["status"] == "PASS" and all(v is True for k, v in gates.items() if k != "fingerprint_crossings")
              and gates["fingerprint_crossings"] == 0, "partition_integrity_report.json")
cap = role_capacity_checks(gate, integ)
L.assert_true("S4.5.capacities", "Sec. 4.5", "gate role totals equal the exact role capacities; all 40 cells pass",
              cap["role_totals_equal_capacities"] and cap["total_equals_pure_benign_rows"] and cap["all_cells_pass"], "multiplicity_gate.csv")
L.check("S4.5.max_dev", "Sec. 4.5", "largest role-by-band deviation (rows)", cap["max_abs_deviation"], "30", "multiplicity_gate.csv")
L.assert_true("S4.5.max_dev_tolerances", "Sec. 4.5", "the two 30-row deviations sit in cells with tolerances 49 and 99",
              sorted(c["tolerance"] for c in cap["cells_at_max_deviation"]) == [49, 99], "multiplicity_gate.csv")
sc = source_checks(read_csv(art(*F, "source_gate_actual.csv")), read_csv(art(*F, "singleton_source_gate.csv")),
                   read_csv(art(*F, "source_x_multiplicity_gate.csv")), gate)
L.assert_true("S4.5.source_gates", "Sec. 4.5", "source totals, singleton-source and source-by-band gates consistent; all non-exempt cells pass",
              sc["source_totals_equal_gate_totals"] and sc["singleton_source_equals_gate_band1"] and sc["source_x_band_sums_equal_gate"]
              and sc["nonexempt_cells_pass"], "source_*_gate.csv")
L.assert_true("S4.5.exemptions", "Sec. 4.5", "exempt cells are exactly UNSW-NB15_1/_2 x >=100 (as frozen in the amendment)",
              sorted((c["source_file"], c["size_band"]) for c in sc["exempt_cells"]) ==
              sorted((c["source_file"], c["size_band"]) for c in load_json(art(*F, "source_multiplicity_amendment.json"))["exempt_source_x_band_cells"]),
              "source_x_multiplicity_gate.csv + source_multiplicity_amendment.json")
for r in ROLES:
    L.check(f"S4.5.singleton_share.{r}", "Sec. 4.5", f"singleton share of pure-benign rows, {r}", 100 * rows.loc[r, "1"] / rows.loc[r].sum(), "84.87", "multiplicity_gate.csv")
tv = {r: tv_distance(rows.loc[r], pool) for r in ROLES}
L.check("T8.final.tv_dev", "Table 8; Sec. 4.5", "TV(development, pool), v3.4 (recomputed from the gate)", tv["development"], "0.000150", "multiplicity_gate.csv")
L.check("T8.final.tv_dev_rounded", "Table 8", "TV(development, pool), v3.4, as tabulated", tv["development"], "0.00015", "multiplicity_gate.csv")
others = [tv[r] for r in ROLES if r != "development"]
L.check("S4.5.tv_min_other", "Sec. 4.5", "smallest TV of the other four roles", min(others), "0.000012", "multiplicity_gate.csv")
L.check("S4.5.tv_max_other", "Sec. 4.5", "largest TV of the other four roles", max(others), "0.000144", "multiplicity_gate.csv")
L.note("S4.5.tv_frozen", "Sec. 4.5; abstract", "frozen-record TV 0.000149 (manuscript also gives the recomputed 0.000150)",
       f"{tv['development']:.6f}", REC, "INFO")
L.note("S4.5.redundancy_rates", "Sec. 4.5", "benign-only role redundancy rates 12.583/12.666/12.521%",
       "not recomputable: per-role group counts outside the >=100 band are not in the evidence set", REC, "NOT_REPRODUCIBLE_HERE")
b = band_ge100_checks(band, gate)
L.assert_true("S2.ge100_rows", "Table S2; Sec. 4.5", ">=100 band rows by role equal the gate", b["rows_equal_gate"], "band_ge100_role_report.csv")
L.check("S4.5.ge100_groups", "Sec. 4.5", "groups in the >=100 band", b["groups_total"], "276", "band_ge100_role_report.csv")
for r, p in zip(ROLES, ["25,751", "5,151", "7,725", "5,127", "7,725"]):
    L.check(f"S4.5.ge100_rows.{r}", "Sec. 4.5", f">=100 band rows, {r}", rows.loc[r, ">=100"], p, "multiplicity_gate.csv")
for r, p in zip(ROLES, ["713", "252", "760", "350", "456"]):
    L.check(f"S2.largest.{r}", "Table S2", f"largest >=100 group, {r}", b["largest_group_by_role"][r], p, "band_ge100_role_report.csv")
L.check("S4.5.wt_ge100_groups", "Sec. 4.5", "weight-training groups in the >=100 band", int(band.set_index("partition").loc["benign_weight_train", "groups"]), "118", "band_ge100_role_report.csv")
L.check("S4.5.wt_gt400", "Sec. 4.5", "weight-training groups larger than 400 rows", integ["weight_train_groups_gt400_in_band_ge100"], "12", "partition_integrity_report.json")
solver = load_json(art(*F, "coordinated_feasibility_solver_manifest.json"))
st = {s["stage"]: s for s in solver["subproblems"]}
L.assert_true("M.solver", "Methods (MILP)", "bands 2-99 solved to optimality; >=100 stage stopped at its time limit; both independently verified feasible",
              st["coordinated_remainder_2_to_99"]["success"] and "Optimal" in st["coordinated_remainder_2_to_99"]["message"]
              and "Time limit" in st["top_ge100"]["message"] and solver["all_subproblems_independently_verified_feasible"]
              and solver["scipy_version"] == "1.16.3", "coordinated_feasibility_solver_manifest.json")

# prevalence of the evaluation roles (development and final-test archives)
dev = read_csv(art("unsw_nb15", "model_selection", "derived_from_archives", "run_summary_development.csv")).iloc[0]
fin = read_csv(art("unsw_nb15", "final_evaluation", "derived_from_archives", "run_summary_final_test.csv")).iloc[0]
L.check("S4.5.prev_dev", "Sec. 4.5", "attack prevalence, UNSW-NB15 development", 100 * dev.n_attack_rows / dev.n_rows, "36.6772", "run_summary_development.csv")
L.check("S4.5.prev_final", "Sec. 4.5", "attack prevalence, UNSW-NB15 final test", 100 * fin.n_attack_rows / fin.n_rows, "36.6773", "run_summary_final_test.csv")

# membership samples (first 100,000 rows of each file)
mu = membership_checks(read_csv(art(*F, "partition_membership_first100k.csv")))
L.assert_true("S4.2.membership_sample_unsw", "Sec. 4.2", "first 100k UNSW-NB15 v3.4 membership rows: unique row_id, no fingerprint in two states, benign-only roles pure",
              mu["row_id_unique"] and mu["fingerprints_in_more_than_one_state"] == 0 and mu["attack_rows_in_benign_only_roles"] == 0
              and mu["mixed_group_rows_in_benign_only_roles"] == 0, "partition_membership_first100k.csv (sample)")

# provenance chain of the UNSW-NB15 allocation regimes
sbm = load_json(art(*SB, "smoke_manifest_source_balanced.json"))
fnm = load_json(art(*F, "smoke_manifest_final.json"))
amd = load_json(art(*F, "source_multiplicity_amendment.json"))
d00c = load_json(art(*SB, "diagnostic_manifest_00C_block_decomposition.json"))
d00d = load_json(art(*SB, "diagnostic_manifest_00D_mixed_label.json"))
sel = load_json(art("unsw_nb15", "model_selection", "selected_models_manifest.json"))
chain = [amd["parent_membership_sha256"] == sbm["upstream_identity"]["partition_manifest_file_sha256"] == integ["parent_membership_sha256"]
         == d00d["partition_manifest_file_sha256"],
         integ["partition_membership_sha256"] == fnm["upstream_identity"]["partition_manifest_file_sha256"]
         == sel["upstream_identity"]["partition_manifest_file_sha256"],
         amd["evidence_00D_manifest_hash"] == d00d["diagnostic_manifest_hash"],
         d00d["parent_00c_manifest_hash"] == d00c["diagnostic_manifest_hash"],
         integ["scientific_policy_hash"] == amd["policy_hash"],
         sbm["final_status"] == "FAIL" and fnm["final_status"] == "PASS"]
L.assert_true("X.unsw_provenance_chain", "Sec. 4.3-4.5; Supplementary S1",
              "source-balanced membership -> 00C/00D diagnostics -> amendment -> v3.4 membership -> PASS gate -> selected models (hash chain)",
              all(chain), "smoke, diagnostic, amendment, integrity and selection manifests")

### CICIDS2017 (Section 4.6)

In [ ]:
ci = cicids_integrity_summary(load_json(art("cicids2017", "partition", "partition_integrity_report.json")))
L.assert_true("S4.6.hard_checks", "Sec. 4.6", "all twenty hard integrity checks passed", ci["status"] == "PASS" and ci["hard_checks"] == 20
              and ci["hard_checks_passed"] == 20 and ci["blocking_failures"] == 0, "cicids2017 partition_integrity_report.json")
ba = read_csv(art("cicids2017", "partition", "benign_allocation_table.csv"))
L.assert_true("S4.6.benign_targets", "Sec. 4.6", "all five benign row targets met exactly", bool((ba.absolute_row_deviation == 0).all()) and len(ba) == 5,
              "benign_allocation_table.csv")
L.check("S4.6.reserve", "Sec. 4.6", "pure-attack rows placed in the unused reserve", ci["attack_reserve_rows"], "11,657", "partition_integrity_report.json")
am = load_json(art("cicids2017", "partition", "attack_allocation_manifest.json"))
L.assert_true("S4.6.reserve_singletons", "Sec. 4.6", "reserve consists of 11,657 groups, i.e. singleton fingerprints only",
              am["reserve_reason_counts_by_group"] == {"BENIGN_MAJORITY_CAP": 11657} and am["attacks_reserved"] == 11657, "attack_allocation_manifest.json")
L.check("S4.6.prev_dev", "Sec. 4.6", "attack prevalence, CICIDS2017 development", 100 * ci["development_attack_prevalence"], "48.9999", "partition_integrity_report.json")
L.check("S4.6.prev_final", "Sec. 4.6", "attack prevalence, CICIDS2017 final test", 100 * ci["final_test_attack_prevalence"], "48.9999", "partition_integrity_report.json")
mc = membership_checks(read_csv(art("cicids2017", "partition", "partition_membership_first100k.csv")))
L.assert_true("S4.6.membership_sample", "Sec. 4.6", "first 100k CICIDS2017 membership rows: unique row_id, no fingerprint in two states, benign-only roles pure",
              mc["row_id_unique"] and mc["fingerprints_in_more_than_one_state"] == 0 and mc["attack_rows_in_benign_only_roles"] == 0
              and mc["mixed_group_rows_in_benign_only_roles"] == 0, "partition_membership_first100k.csv (sample)")
L.note("S4.6.early_stop", "Sec. 4.6", "early-stopping role: 227,309 rows, 55,372 distinct vectors (75.64% redundant)",
       "not recomputable: the full CICIDS2017 membership is not in the evidence set", REC, "NOT_REPRODUCIBLE_HERE")

### outputs

In [ ]:
t8 = pd.DataFrame([
    {"regime": "Initial fingerprint-atomic", "fingerprint_overlap": "0", "development_multiplicity_tv": "not available",
     "development_benign_exceedance_pct": "99.22", "calibration_benign_exceedance_pct": "not available", "source": REC},
    {"regime": "Source-balanced", "fingerprint_overlap": "0 (analysis record)", "development_multiplicity_tv": f"{tv_sb:.4f}",
     "development_benign_exceedance_pct": f"{100 * sbm['development_benign_exceedance']:.2f}",
     "calibration_benign_exceedance_pct": f"{100 * sbm['calibration_benign_exceedance']:.4f}",
     "source": "recomputed: source_balanced/{development_benign_fingerprint_level_summary.csv,smoke_manifest_source_balanced.json}"},
    {"regime": "Source- and multiplicity-balanced (v3.4)", "fingerprint_overlap": str(gates["fingerprint_crossings"]),
     "development_multiplicity_tv": f"{tv['development']:.6f}",
     "development_benign_exceedance_pct": f"{100 * fnm['development_benign_exceedance']:.4f}",
     "calibration_benign_exceedance_pct": f"{100 * fnm['calibration_benign_exceedance']:.4f}",
     "source": "recomputed: final_source_multiplicity_balanced/{multiplicity_gate.csv,partition_integrity_report.json,smoke_manifest_final.json}"}])
write_csv(t8, RESULTS / "tables" / "Table_3_2_UNSW_structural_validation.csv")
ge = band.merge(gate[gate.size_band == ">=100"][["partition", "target_rows", "actual_rows", "tolerance"]], on="partition")
write_csv(ge[["partition", "target_rows", "actual_rows", "tolerance", "groups", "mean_group_size", "max_group_size"]],
          RESULTS / "tables" / "Table_3_3_ge100_band_extract.csv")
write_csv(gate, RESULTS / "tables" / "Table_S2_role_by_band_allocation.csv")
write_csv(pd.DataFrame({"role": list(tv), "tv_to_pool": list(tv.values()),
                        "singleton_share_pct": [100 * rows.loc[r, "1"] / rows.loc[r].sum() for r in tv]}),
          RESULTS / "tables" / "Table_3_2c_role_tv_distances.csv")
assert L.finish() == 0, "some checks FAILED - see the claim log below"

## Results

In [ ]:
# ===== Results of the checks =====
import pandas as pd
from IPython.display import display
log = pd.read_csv(REPO_ROOT / "results" / "claims_check" / "reproduce_partitions.csv")
print(log.status.value_counts().to_string())
display(log[log.status != "PASS"])          # notes: values that cannot be recomputed here, or corrections
display(log.head(15))                        # first passing checks (all rows are in the CSV)

In [ ]:
# ===== Tables written by this notebook =====
import pandas as pd
from IPython.display import display
pd.set_option('display.max_columns', 30)
print("results/tables/Table_3_2_UNSW_structural_validation.csv"); display(pd.read_csv(REPO_ROOT / "results/tables/Table_3_2_UNSW_structural_validation.csv"))
print("results/tables/Table_3_2b_source_balanced_profile.csv"); display(pd.read_csv(REPO_ROOT / "results/tables/Table_3_2b_source_balanced_profile.csv"))
print("results/tables/Table_3_2c_role_tv_distances.csv"); display(pd.read_csv(REPO_ROOT / "results/tables/Table_3_2c_role_tv_distances.csv"))
print("results/tables/Table_3_3_ge100_band_extract.csv"); display(pd.read_csv(REPO_ROOT / "results/tables/Table_3_3_ge100_band_extract.csv"))